# W2-D1: Alert Correlation

Three-layer pipeline:
1. dedup: collapse repeated fires of the same alert type via fingerprint
2. session window: group alerts that arrive within `gap_sec` of each other
3. topology: within each session, merge services that are directly connected on the service graph

In [ ]:
import json
import os
from collections import defaultdict
from dateutil.parser import parse
import networkx as nx

SEVERITY_ORDER = {"warn": 0, "crit": 1}


def fingerprint(alert: dict) -> str:
    # stable identity of an alert type
    # timestamp and value are excluded so repeated fires share the same fingerprint
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"


def dedup(alerts: list) -> tuple[list, dict]:
    # collapse repeated fires: keep the first occurrence of each fingerprint
    # return (deduped alert list, mapping of fingerprint -> all original alert ids)
    fp_map: dict[str, list] = defaultdict(list)
    seen: dict[str, dict] = {}
    for a in sorted(alerts, key=lambda x: x["ts"]):
        fp = fingerprint(a)
        fp_map[fp].append(a["id"])
        if fp not in seen:
            seen[fp] = a
    return list(seen.values()), dict(fp_map)


def session_groups(alerts: list, gap_sec: int = 120) -> list:
    # split a sorted alert stream into sessions
    # a new session starts when the gap to the previous alert exceeds gap_sec
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a["ts"])
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        last_ts = parse(groups[-1][-1]["ts"])
        if (parse(alert["ts"]) - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups


def topology_group(alerts: list, graph: nx.DiGraph) -> list:
    # restrict the graph to only the services present in this alert set
    # then return each connected component as a group
    # this avoids chaining through non-alerting intermediary nodes
    alert_services = {a["service"] for a in alerts}
    subgraph = graph.subgraph(alert_services).to_undirected()
    by_service: dict[str, list] = defaultdict(list)
    for a in alerts:
        by_service[a["service"]].append(a)
    result = []
    for component in nx.connected_components(subgraph):
        group = []
        for svc in component:
            group.extend(by_service[svc])
        result.append(group)
    return result


def correlate(
    alerts: list,
    graph: nx.DiGraph,
    gap_sec: int = 120,
) -> list:
    # layer 1: dedup
    deduped, fp_all_ids = dedup(alerts)

    # layer 2: session window on the deduped set
    sessions = session_groups(deduped, gap_sec=gap_sec)

    clusters = []
    for s_idx, session_alerts in enumerate(sessions):
        # layer 3: topology grouping within each session
        for g_idx, group in enumerate(topology_group(session_alerts, graph)):
            fps = sorted({fingerprint(a) for a in group})
            # expand back to all original alert ids for this cluster
            all_ids = sorted(
                aid for fp in fps for aid in fp_all_ids.get(fp, [])
            )
            clusters.append({
                "cluster_id": f"c-{s_idx:03d}-{g_idx:03d}",
                "alert_count": len(all_ids),
                "services": sorted({a["service"] for a in group}),
                "time_range": [
                    min(a["ts"] for a in group),
                    max(a["ts"] for a in group),
                ],
                "max_severity": max(
                    (a["severity"] for a in group),
                    key=lambda s: SEVERITY_ORDER.get(s, 0),
                ),
                "fingerprints": fps,
                "alert_ids": all_ids,
            })
    return clusters


print("helpers loaded")

helpers loaded


In [ ]:
# load alerts (plain json parse, no extra dependencies)
alerts = []
with open("dataset/alerts_sample.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))

print(f"loaded {len(alerts)} alerts")
print(f"time span: {alerts[0]['ts']} to {alerts[-1]['ts']}")

loaded 20 alerts
time span: 2026-06-12T09:42:01Z to 2026-06-12T09:48:30Z


In [ ]:
# build service dependency graph
with open("dataset/services.json", encoding="utf-8") as f:
    svc_data = json.load(f)

graph = nx.DiGraph()
for svc in svc_data["services"]:
    graph.add_node(svc["name"], **{k: v for k, v in svc.items() if k != "name"})
for edge in svc_data["edges"]:
    graph.add_edge(edge["from"], edge["to"], type=edge["type"])

print(f"graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# show neighbors for each service that has alerts
alert_services = {a["service"] for a in alerts}
print(f"\nservices in alert set: {sorted(alert_services)}")
for svc in sorted(alert_services):
    if graph.has_node(svc):
        neighbors = list(graph.successors(svc)) + list(graph.predecessors(svc))
        print(f"  {svc} neighbors: {neighbors}")

graph: 14 nodes, 17 edges

services in alert set: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
  cart-svc neighbors: ['cart-redis', 'catalog-svc', 'checkout-svc']
  checkout-svc neighbors: ['cart-svc', 'payment-svc', 'inventory-svc', 'notification-svc', 'edge-lb']
  edge-lb neighbors: ['auth-svc', 'catalog-svc', 'search-svc', 'checkout-svc']
  notification-svc neighbors: ['kafka-events', 'checkout-svc']
  payment-svc neighbors: ['payments-db', 'checkout-svc']
  recommender-svc neighbors: ['catalog-db', 'catalog-svc']
  search-svc neighbors: ['catalog-db', 'edge-lb']


In [ ]:
# run correlation pipeline
# gap_sec=120: all 20 alerts span about 6.5 minutes, so 120s keeps them in one session
# and lets topology do the separation work
GAP_SEC = 120

clusters = correlate(alerts, graph, gap_sec=GAP_SEC)

input_count = len(alerts)
output_count = len(clusters)
reduction_ratio = round(1 - output_count / input_count, 4)

print(f"input alerts  : {input_count}")
print(f"output clusters: {output_count}")
print(f"reduction ratio: {reduction_ratio}")
print()
for c in clusters:
    print(
        f"[{c['cluster_id']}] {c['alert_count']} alerts"
        f" | severity={c['max_severity']}"
        f" | services={c['services']}"
        f" | range={c['time_range']}"
    )

input alerts  : 20
output clusters: 2
reduction ratio: 0.9

[c-000-000] 19 alerts | severity=crit | services=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'search-svc'] | range=['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z']
[c-000-001] 1 alerts | severity=warn | services=['recommender-svc'] | range=['2026-06-12T09:45:10Z', '2026-06-12T09:45:10Z']


In [ ]:
# write results/cluster_summary.json
os.makedirs("results", exist_ok=True)

summary = {
    "input_alerts": input_count,
    "output_clusters": output_count,
    "reduction_ratio": reduction_ratio,
    "clusters": clusters,
}

with open("results/cluster_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("wrote results/cluster_summary.json")

# validate acceptance criteria from the assignment
assert os.path.exists("results/cluster_summary.json"), "output file missing"
with open("results/cluster_summary.json", encoding="utf-8") as f:
    loaded = json.load(f)
assert loaded["reduction_ratio"] >= 0.5, "reduction_ratio below 0.5"
for c in loaded["clusters"]:
    assert c["services"], "cluster missing services"
    assert len(c["time_range"]) == 2, "cluster missing time_range"
print(f"all acceptance criteria passed - {output_count} clusters, ratio={reduction_ratio}")